## Data Loading - International Football Results 1872 to 2026

Loading in data using kaggle. Data used: https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017/data

This dataset includes 49,393 results of international football matches starting from the very first official match in 1872 up to 2025. The matches range from FIFA World Cup to FIFI Wild Cup to regular friendly matches. The matches are strictly men's full internationals and the data does not include Olympic Games or matches where at least one of the teams was the nation's B-team, U-23 or a league select team.

In [155]:
import shutil
import kagglehub
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path

DATA_DIRECTORY = Path("../data/international-football-results")

def get_data_path(data_directory:Path) -> Path:

    if data_directory.exists():
        path = data_directory
        print("Using cached dataset at:", path)
    else:
        download_path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")
        data_directory.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(download_path, data_directory)
        path = data_directory
        print("Downloaded dataset to:", path)

    return path


def load_data(data_directory: Path) -> dict[str,pd.DataFrame]:
    return {file_path.stem : pd.read_csv(file_path) for file_path in sorted(data_directory.glob('*csv'))}


frames = load_data(DATA_DIRECTORY)
results = frames['results']
results['date'] = pd.to_datetime(results['date'])
results['year'] = results['date'].dt.year

shootouts = frames['shootouts']
goalscorers = frames['goalscorers']
former_names = frames['former_names']

In [146]:
len(results[results['home_score'].isna()])

not_played = results[results['home_score'].isna()]
played = results[~results['home_score'].isna()]

played.head()

match_conditions = [
    played['home_score'] > played['away_score'],
    played['home_score'] < played['away_score']
]  

choices = [1,-1]
played['result'] = np.select(match_conditions,choices,default=0)


In [ ]:
played['goal_diff'] = np.abs(played['home_score']-played['away_score'])

played_long = played.melt(
    id_vars=['year','tournament'],
    value_vars=['home_team','away_team'],
    value_name='team'
)

wc_matches = played_long[(played_long['tournament'] == 'FIFA World Cup') & (played_long['year'] == 2026)]
wc_teams = wc_matches['team'].unique()
wc_former_names = former_names[former_names['current'].isin(wc_teams)]

name_map = dict(zip(wc_former_names['former'], wc_former_names['current']))


played['home_team'] = played['home_team'].replace(name_map)
played['away_team'] = played['away_team'].replace(name_map)




48
